In [1]:

import pandas as pd
import numpy as np
import os
import re
from tqdm import tqdm

tqdm.pandas()

# =========================================
# 1. PATHS
# =========================================

BASE_PATH = r"C:\Users\Neda\Desktop\personality_llm"

PROCESSED_DATA_PATH = os.path.join(BASE_PATH, "data", "processed")
FEATURES_DATA_PATH = os.path.join(BASE_PATH, "data", "features")

os.makedirs(FEATURES_DATA_PATH, exist_ok=True)

TRAIN_PATH = os.path.join(PROCESSED_DATA_PATH, "train_clean.csv")
TEST_PATH = os.path.join(PROCESSED_DATA_PATH, "test_clean.csv")
KAMTERA_PATH = os.path.join(PROCESSED_DATA_PATH, "kamtera_clean.csv")

print("PROCESSED:", PROCESSED_DATA_PATH)
print("FEATURES:", FEATURES_DATA_PATH)

# =========================================
# 2. LOAD DATA
# =========================================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
kamtera_df = pd.read_csv(KAMTERA_PATH)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Kamtera:", kamtera_df.shape)

# =========================================
# 3. FEATURE DICTIONARIES (IMPROVED)
# =========================================

POS_WORDS = {"خوب","عالی","دوست","موفق","زیبا","خوشحال","امید","مثبت","رضایت","حمایت"}
NEG_WORDS = {"بد","ضعیف","مشکل","ناراحت","نگران","ترس","استرس","منفی","اشتباه","بحران"}

FIRST_PERSON = {"من","خودم","ما","برایم","برام","به‌نظرم","فکر","می‌کنم"}
SOCIAL = {"دوست","خانواده","همکار","گروه","مردم","دیگران","کمک","حمایت","تیم"}
CERTAINTY = {"حتما","قطعاً","مطمئن","یقیناً","بدون شک"}
UNCERTAINTY = {"شاید","احتمالاً","ممکن","نمی‌دانم","فکر کنم","ظاهراً"}

# =========================================
# 4. SAFE COUNT FUNCTION
# =========================================

def count_words(text, word_set):
    text = str(text)
    return sum(1 for w in word_set if w in text)

# =========================================
# 5. CORE LEXICAL FEATURES
# =========================================

def lexical_diversity(text):
    words = str(text).split()
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

def sentence_count(text):
    return len([s for s in re.split(r"[.!؟?]+", str(text)) if s.strip()])

def avg_sentence_length(text):
    sentences = [s for s in re.split(r"[.!؟?]+", str(text)) if s.strip()]
    if len(sentences) == 0:
        return 0
    return np.mean([len(s.split()) for s in sentences])

def repeated_chars(text):
    return len(re.findall(r"(.)\1{2,}", str(text)))

# =========================================
# 6. FEATURE ENGINEERING (CLEAN VERSION)
# =========================================

def extract_features(text):
    text = str(text)
    words = text.split()
    n_words = len(words)

    if n_words == 0:
        n_words = 1  # avoid division by zero

    return pd.Series({
        # lexical
        "lexical_diversity": lexical_diversity(text),
        "sentence_count": sentence_count(text),
        "avg_sentence_length": avg_sentence_length(text),
        "repeated_char_count": repeated_chars(text),

        # sentiment proxy
        "pos_count": count_words(text, POS_WORDS),
        "neg_count": count_words(text, NEG_WORDS),

        # social/psychological
        "first_person": count_words(text, FIRST_PERSON),
        "social_words": count_words(text, SOCIAL),

        # certainty
        "certainty": count_words(text, CERTAINTY),
        "uncertainty": count_words(text, UNCERTAINTY),

        # ratios
        "pos_ratio": count_words(text, POS_WORDS) / n_words,
        "neg_ratio": count_words(text, NEG_WORDS) / n_words,
        "first_person_ratio": count_words(text, FIRST_PERSON) / n_words,
        "social_ratio": count_words(text, SOCIAL) / n_words,
        "certainty_ratio": count_words(text, CERTAINTY) / n_words,
        "uncertainty_ratio": count_words(text, UNCERTAINTY) / n_words,
    })

# =========================================
# 7. APPLY FEATURES
# =========================================

print("Extracting train features...")
train_feat = train_df["clean_text"].progress_apply(extract_features)
train_df = pd.concat([train_df, train_feat], axis=1)

print("Extracting test features...")
test_feat = test_df["clean_text"].progress_apply(extract_features)
test_df = pd.concat([test_df, test_feat], axis=1)

print("Extracting Kamtera features...")
kamtera_feat = kamtera_df["clean_text"].progress_apply(extract_features)
kamtera_df = pd.concat([kamtera_df, kamtera_feat], axis=1)

# =========================================
# 8. FEATURE SET (FINAL CLEAN LIST)
# =========================================

FEATURE_COLUMNS = [
    "lexical_diversity",
    "sentence_count",
    "avg_sentence_length",
    "repeated_char_count",

    "pos_count",
    "neg_count",
    "first_person",
    "social_words",
    "certainty",
    "uncertainty",

    "pos_ratio",
    "neg_ratio",
    "first_person_ratio",
    "social_ratio",
    "certainty_ratio",
    "uncertainty_ratio"
]

print("Total features:", len(FEATURE_COLUMNS))

# =========================================
# 9. LABEL DISTRIBUTION CHECK (IMPORTANT)
# =========================================

print("\nLabel distribution:")
print(train_df["label"].value_counts().sort_index())

train_df["label"].value_counts().to_csv(
    os.path.join(FEATURES_DATA_PATH, "label_distribution.csv"),
    encoding="utf-8-sig"
)

# =========================================
# 10. SAVE FULL DATASETS
# =========================================

train_df.to_csv(os.path.join(FEATURES_DATA_PATH, "train_features.csv"), index=False, encoding="utf-8-sig")
test_df.to_csv(os.path.join(FEATURES_DATA_PATH, "test_features.csv"), index=False, encoding="utf-8-sig")
kamtera_df.to_csv(os.path.join(FEATURES_DATA_PATH, "kamtera_features.csv"), index=False, encoding="utf-8-sig")

# =========================================
# 11. MODEL-READY MATRICES
# =========================================

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df["label"]

X_test = test_df[FEATURE_COLUMNS]

X_train.to_csv(os.path.join(FEATURES_DATA_PATH, "X_train.csv"), index=False)
y_train.to_csv(os.path.join(FEATURES_DATA_PATH, "y_train.csv"), index=False)
X_test.to_csv(os.path.join(FEATURES_DATA_PATH, "X_test.csv"), index=False)

# =========================================
# 12. FINAL SUMMARY
# =========================================

print("\n====================")
print("FINAL SUMMARY")
print("====================")
print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Kamtera:", kamtera_df.shape)
print("Features:", len(FEATURE_COLUMNS))

print("\nDONE ✔")

PROCESSED: C:\Users\Neda\Desktop\personality_llm\data\processed
FEATURES: C:\Users\Neda\Desktop\personality_llm\data\features
Train: (800, 12)
Test: (800, 11)
Kamtera: (107280, 11)
Extracting train features...


100%|██████████| 800/800 [00:00<00:00, 3003.38it/s]


Extracting test features...


100%|██████████| 800/800 [00:00<00:00, 2648.70it/s]


Extracting Kamtera features...


100%|██████████| 107280/107280 [00:32<00:00, 3263.20it/s]


Total features: 16

Label distribution:
label
1    131
2    139
3    135
4    137
5    132
6    126
Name: count, dtype: int64

FINAL SUMMARY
Train: (800, 28)
Test: (800, 27)
Kamtera: (107280, 27)
Features: 16

DONE ✔
